In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import os 
import nltk

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\CANDRAGNAWN\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\CANDRAGNAWN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:

!pip install -q textstat
!pip install -q googletrans
!pip install contractions


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
DATA_PATH = "dataset/"
os.listdir(DATA_PATH)

['sample_submission.csv', 'test.csv', 'test_labels.csv', 'train.csv']

In [4]:
TEST_PATH = os.path.join(DATA_PATH, "test.csv")
TEST_LABELS_PATH = os.path.join(DATA_PATH, "test.labels.csv")
SAMPLE_PATH = os.path.join(DATA_PATH, "sample_submission.csv")
TRAIN_PATH = os.path.join(DATA_PATH, "train.csv")
test_data = pd.read_csv(TEST_PATH)
train_data = pd.read_csv(TRAIN_PATH)
test_labels = pd.read_csv(TEST_LABELS_PATH) if os.path.exists(TEST_LABELS_PATH) else None

In [5]:
df = pd.read_csv('dataset/train.csv')
df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [6]:
from EDA.exploratory_data import exploratory_data_analysis
df['label'] = df[['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']].max(axis=1)
exploratory_data_analysis(df)

Data Shape: (159571, 9)

Data Types:
 id                 str
comment_text       str
toxic            int64
severe_toxic     int64
obscene          int64
threat           int64
insult           int64
identity_hate    int64
label            int64
dtype: object

Missing Values:
 id               0
comment_text     0
toxic            0
severe_toxic     0
obscene          0
threat           0
insult           0
identity_hate    0
label            0
dtype: int64

Statistical Summary:
                toxic   severe_toxic        obscene         threat  \
count  159571.000000  159571.000000  159571.000000  159571.000000   
mean        0.095844       0.009996       0.052948       0.002996   
std         0.294379       0.099477       0.223931       0.054650   
min         0.000000       0.000000       0.000000       0.000000   
25%         0.000000       0.000000       0.000000       0.000000   
50%         0.000000       0.000000       0.000000       0.000000   
75%         0.000000       0.0000

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,label
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
159566,ffe987279560d7ff,""":::::And for the second time of asking, when ...",0,0,0,0,0,0,0
159567,ffea4adeee384e90,You should be ashamed of yourself \n\nThat is ...,0,0,0,0,0,0,0
159568,ffee36eab5c267c9,"Spitzer \n\nUmm, theres no actual article for ...",0,0,0,0,0,0,0
159569,fff125370e4aaaf3,And it looks like it was actually you who put ...,0,0,0,0,0,0,0


In [7]:
from preprocessing.text_preprocessing import TextPreprocessor
from preprocessing.text_preprocessing import TextPreprocessorNonStopword
preprocessor = TextPreprocessor()
preprocessor_nonstopword = TextPreprocessorNonStopword()
df['clean_comment_text'] = df['comment_text'].apply(
    preprocessor.preprocess_text
)

df['nonstopword_comment_text'] = df['comment_text'].apply(
    preprocessor_nonstopword.preprocess_text_non_stopword
)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\CANDRAGNAWN\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\CANDRAGNAWN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\CANDRAGNAWN\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Labelin Data

In [ ]:
#definisi kolom di dataset
label_columns = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

# jika ada di dalam kolom itu nilai nya 1 pasti toxic
df['is_toxic'] = df[label_columns].max(axis=1)
X1 = df['clean_comment_text']
X2 = df['nonstopword_comment_text']
Y = df['is_toxic']

print("Distribusi Target Y (0 = Non-Toxic, 1 = Toxic):")
print(Y.value_counts())

Distribusi Target Y (0 = Non-Toxic, 1 = Toxic):
is_toxic
0    143346
1     16225
Name: count, dtype: int64


In [17]:
df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,label,clean_comment_text,nonstopword_comment_text,is_toxic
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0,0,explanation edits made username hardcore metal...,explanation why the edits made under my userna...,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0,0,daww match background colour seemingly stuck t...,daww he match this background colour i am seem...,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0,0,hey man really trying edit war guy constantly ...,hey man i am really not trying to edit war it ...,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0,0,make real suggestion improvement wondered sect...,more i can not make any real suggestion on imp...,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0,0,sir hero chance remember page,you sir are my hero any chance you remember wh...,0


TF-IDF

In [ ]:
    from features.tfidf_extractor import TextVectorizer
    from sklearn.model_selection import train_test_split

    #pake stop word
    X1_train, X1_test, Y_train, Y_test = train_test_split(X1, Y, test_size=0.2, random_state=42)

    vectorizer = TextVectorizer(max_features=5000)
    X1_train_tfidf = vectorizer.fit_transform(X1_train)
    X1_test_tfidf = vectorizer.transform(X1_test)

    print("Shape X1_train TF-IDF:", X1_train_tfidf.shape)

    #non stopword
    X2_train, X2_test, Y2_train, Y2_test = train_test_split(X2, Y, test_size=0.2, random_state=42)

    vectorizer_nonstopword = TextVectorizer(max_features=5000)
    X2_train_tfidf = vectorizer_nonstopword.fit_transform(X2_train)
    X2_test_tfidf = vectorizer_nonstopword.transform(X2_test)

    print("Shape X2_train TF-IDF:", X2_train_tfidf.shape)

Shape X1_train TF-IDF: (127656, 5000)
Shape X2_train TF-IDF: (127656, 5000)


LOGISTIC REGRRESSION (MODEL)

In [10]:
from models.logistic_model import ToxicClassifier

classifier = ToxicClassifier()
classifier_nonstopword = ToxicClassifier()

#stopword
classifier.train(X1_train_tfidf, Y_train)
#nonstopword
classifier_nonstopword.train(X2_train_tfidf, Y_train)

#stopword
Y_pred = classifier.predict(X1_test_tfidf)
#nonstopword
Y2_pred = classifier_nonstopword.predict(X2_test_tfidf)

#hasil
report1 = classifier.evaluate(Y_test, Y_pred)
report2 = classifier_nonstopword.evaluate(Y_test, Y2_pred)

print("STOP WORD")
print(report1)
print("NON STOP WORD")
print(report2)

STOP WORD
              precision    recall  f1-score   support

           0       0.96      0.99      0.98     28671
           1       0.92      0.63      0.75      3244

    accuracy                           0.96     31915
   macro avg       0.94      0.81      0.86     31915
weighted avg       0.96      0.96      0.95     31915

NON STOP WORD
              precision    recall  f1-score   support

           0       0.96      0.99      0.98     28671
           1       0.91      0.62      0.74      3244

    accuracy                           0.96     31915
   macro avg       0.93      0.81      0.86     31915
weighted avg       0.95      0.96      0.95     31915



In [11]:
words = vectorizer_nonstopword.get_feature_names() 
coeffs = classifier_nonstopword.get_coefficients() 

df_indikator = pd.DataFrame({'Kata': words, 'Bobot': coeffs})
bobot_hell = df_indikator[df_indikator["Kata"] == "hell"]
print("\n" + "="*50)
print("10 INDIKATOR KATA TOXIC")
print(df_indikator.sort_values(by='Bobot', ascending=False).head(10).to_string(index=False))
print("")
print("10 INDIKATOR KATA NON-TOXIC")
print(df_indikator.sort_values(by='Bobot', ascending=True).head(10).to_string(index=False))
print(bobot_hell)

print("\n" + "="*50)

df_error = pd.DataFrame({
    'Teks': X2_test,
    'Asli': Y_test,
    'Prediksi': Y2_pred
})

fp = df_error[(df_error['Asli'] == 0) & (df_error['Prediksi'] == 1)]
fn = df_error[(df_error['Asli'] == 1) & (df_error['Prediksi'] == 0)]

print("Contoh False Positif")
print(fp['Teks'].head(5).values)
print("")
print("")
print("Contoh False Negative")
print(fn['Teks'].head(5).values)


10 INDIKATOR KATA TOXIC
    Kata     Bobot
    fuck 15.584067
 fucking 14.573534
   idiot 13.805774
    shit 13.285842
  stupid 12.764482
      as 10.916065
 asshole 10.253962
    suck 10.170175
    crap  9.721116
bullshit  9.631683

10 INDIKATOR KATA NON-TOXIC
     Kata     Bobot
   thanks -3.024538
    thank -2.853426
 redirect -2.780580
thank you -2.739555
     talk -2.702719
     best -2.690873
     help -2.555292
      utc -2.318726
    cheer -2.311818
  welcome -2.266951
      Kata     Bobot
1724  hell  6.489545

Contoh False Positif
<StringArray>
[                                                                                                                                                                                                                                                                                                                                                                                                                                    'live free or die 

In [20]:
kalimat_baru = input("Masukkan kalimat yang ingin dites: ")

if kalimat_baru.strip() != "":
    
    kalimat_bersih = preprocessor_nonstopword.preprocess_text_non_stopword(kalimat_baru)

    fitur_baru = vectorizer_nonstopword.transform([kalimat_bersih])

    prediksi = classifier_nonstopword.predict(fitur_baru)

    print("\n" + "="*50)
    print(f"Kalimat Input : '{kalimat_baru}'")
    print("-" * 50)
    if prediksi[0] == 1:
        print(" HASIL: TOKSIK (Negatif/Kasar) ")
    else:
        print(" HASIL: NORMAL (Aman) ")
    print("="*50)

else:
    print("Kalimat tidak boleh kosong!")


Kalimat Input : 'Oh, I didn't realize you were the main character.'
--------------------------------------------------
 HASIL: NORMAL (Aman) 
